In [3]:
# Elasticsearch 연결 & 인덱스 생성 ===
import json, yaml
from dotenv import load_dotenv
from utils.rag_utils_es import get_es_client, create_es_schema, build_elasticsearch_index  # :contentReference[oaicite:0]{index=0}

with open('config/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

load_dotenv()

es = get_es_client()  # .env의 ES_ENDPOINT, ELASTIC_API_KEY 사용

INDEX_PREFIX = ""   # ""면 전체

# 1) 간단히 "이름만" 목록
all_indices = es.indices.get_alias(index="*").keys()
names = sorted([i for i in all_indices if i.startswith(INDEX_PREFIX)])
print("📚 인덱스 이름 목록:")
for n in names:
    print(" -", n)


📚 인덱스 이름 목록:
 - kgs_index
 - susi_index


In [4]:
# 1. 설정 로드 (인덱스 이름을 알기 위해)
target_index = config['elasticsearch']['kgs_index']  # 조회할 인덱스 명

# 2. 클라이언트 연결
es_client = get_es_client()

# 3. 인덱스 정보 조회 (매핑 + 설정)
# ignore=[404]는 인덱스가 없을 경우 에러 대신 None을 반환하게 함
try:
    response = es_client.indices.get(index=target_index)
    
    # 4. 보기 좋게 출력 (한글 깨짐 방지: ensure_ascii=False)
    print(json.dumps(response.body, indent=2, ensure_ascii=False))

except Exception as e:
    print(f"❌ 조회 실패: {e}")

{
  "kgs_index": {
    "aliases": {},
    "mappings": {
      "properties": {
        "metadata": {
          "properties": {
            "category1": {
              "type": "keyword"
            },
            "category2": {
              "type": "keyword"
            },
            "category3": {
              "type": "keyword"
            },
            "question": {
              "type": "text",
              "analyzer": "standard"
            },
            "question_morph": {
              "type": "text",
              "analyzer": "kiwi_ws"
            },
            "text_morph": {
              "type": "text",
              "analyzer": "kiwi_ws"
            },
            "university": {
              "type": "keyword"
            },
            "year": {
              "type": "integer"
            }
          }
        },
        "text": {
          "type": "text",
          "analyzer": "standard"
        },
        "vector_field": {
          "type": "dense_vector",
        

In [5]:
# 인덱스 정보/문서 조회
INDEX_NAME = "kgs_index"

# 2) 스키마(설정/매핑) 구성 후 인덱스 생성(이미 있으면 건너뜀)
body = create_es_schema()
build_elasticsearch_index(es, INDEX_NAME, body)

INDEX_NAME = INDEX_NAME  # 위 셀에서 지정한 이름 재사용

if not es.indices.exists(index=INDEX_NAME):
    raise ValueError(f"인덱스 '{INDEX_NAME}'가 없습니다. 먼저 생성하세요.")

# 1) 문서 수
count = es.count(index=INDEX_NAME)["count"]

# 2) 설정/매핑 일부 보기
settings = es.indices.get_settings(index=INDEX_NAME)[INDEX_NAME]["settings"]
mappings = es.indices.get_mapping(index=INDEX_NAME)[INDEX_NAME]["mappings"]

# 3) 문서 샘플 5건
doc_size = 5
resp = es.search(index=INDEX_NAME, size=doc_size, query={"match_all": {}})
resp


✅ 인덱스 'kgs_index'에 연결 성공


ObjectApiResponse({'took': 1, 'timed_out': False, '_shards': {'total': 3, 'successful': 3, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1302, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'kgs_index', '_id': '31265668a637196cf0d3590589f0585ec9f7f91237cc562f736de76f37477d3c', '_score': 1.0, '_source': {'text': '✅  재외국민특례(3년·12년):\n한국 국적 학생이 해외에서 일정 기간 이상 교육을 받은 경우.\n(즉, 국적은 한국인이면서 학력 요건을 충족해야 함.)\n1)12년특례: 초중고 전교육과정을 해외소재 학교에서 이수\n2)3년특례: 보호자 자격 + 중고교 3년 동안 해외소재 학교 이수 조건 충족해야 함\n\n✅ 외국인전형:\n부모 모두가 외국인(대한민국 국적 X)인 학생이 대상.\n한국 국적 학생은 지원 불가.\n→ 국적 중심 전형이므로 재외국민특례와 완전히 별개임.', 'metadata': {'year': 2026, 'university': '공통', 'category1': '전형이해', 'category2': '3년, 12년특례', 'category3': '외국인전형', 'question': '3년, 12년특례와 외국인전형은 어떤 차이가 있나요?', 'question_morph': '3 년 , 12 년 특례 와 외국인 전형 은 어떤 차이 가 있 나요 ?', 'text_morph': '✅ 재외 국민 특례 ( 3 년 · 12 년 ) : 한국 국적 학생 이 해외 에서 일정 기간 이상 교육 을 받 은 경우 . ( 즉 , 국적 은 한국인 이 면서 학력 요건 을 충족 하 어야 하 ᆷ . ) 1 ) 12 년 특례 : 초중고 전 교육 과정 을 해외 소재 학교 에서 이수 2 ) 3 년 

In [15]:
target = "kgs_index"

if es.indices.exists(index=target):
    es.indices.delete(index=target, ignore_unavailable=True)
    print(f"🗑️ 인덱스 '{target}' 삭제 완료")
else:
    print(f"인덱스 '{target}'는 존재하지 않습니다.")

🗑️ 인덱스 'kgs_index' 삭제 완료


In [ ]:
# Elasticsearch k1, b 인덱스 삭제
from dotenv import load_dotenv
load_dotenv()

from utils.rag_utils_es import get_es_client, create_es_schema, build_elasticsearch_index  # :contentReference[oaicite:0]{index=0}

es = get_es_client()  # .env의 ES_ENDPOINT, ELASTIC_API_KEY 사용

INDEX_PREFIX = ""   # ""면 전체

# 1) 간단히 "이름만" 목록
all_indices = es.indices.get_alias(index="*").keys()
names = sorted([i for i in all_indices if i.startswith(INDEX_PREFIX)])

names.remove('kgs_index')
names.remove('susi_index')

for target in names:
    if es.indices.exists(index=target):
        es.indices.delete(index=target, ignore_unavailable=True)
        print(f"🗑️ 인덱스 '{target}' 삭제 완료")
    else:
        print(f"인덱스 '{target}'는 존재하지 않습니다.")

🗑️ 인덱스 'kgs_index' 삭제 완료
🗑️ 인덱스 'susi_index' 삭제 완료
